# OLS and GLMs with statsmodels

## Overview

`statsmodels` provides R-style formula-based model fitting with full inference output — p-values, confidence intervals, and diagnostics — making it the natural Python choice for confirmatory statistical modelling.

**Model families covered:**

| Model | Response | Link | Use when |
|---|---|---|---|
| OLS | Continuous | Identity | Normal errors, homoscedastic |
| Logistic | Binary 0/1 | Logit | Binary outcome |
| Poisson | Count ≥ 0 | Log | Count data, no overdispersion |
| Negative Binomial | Count ≥ 0 | Log | Count data with overdispersion |
| Gamma | Continuous > 0 | Log | Right-skewed positive continuous |

**Formula interface:** `"y ~ x1 + x2 + x1:x2"` mirrors R's `lm()`/`glm()` syntax exactly.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats

rng = np.random.default_rng(42)
n = 200
elevation   = rng.uniform(50, 400, n)
nitrate     = rng.gamma(2, 2, n)
treatment   = rng.choice([0, 1], n)
catchment   = rng.choice(['North','South','East','West'], n)
richness    = (25 - 0.04*elevation - 0.8*nitrate + 3.0*treatment
               + rng.normal(0, 3.5, n)).clip(0)
restored    = (richness > richness.mean()).astype(int)
invertcount = rng.negative_binomial(
    n=5, p=0.4, size=n) + (treatment * rng.poisson(3, n))
df = pd.DataFrame(dict(elevation=elevation, nitrate=nitrate,
    treatment=treatment, catchment=catchment,
    richness=richness, restored=restored, invertcount=invertcount))
print(df.describe().round(2))

---
## OLS: Linear Regression

In [ ]:
# Formula interface -- same syntax as R's lm()
ols = smf.ols("richness ~ elevation + nitrate + C(treatment) + C(catchment)",
              data=df).fit()
print(ols.summary())
# Confidence intervals
print("\n95% Confidence intervals:")
print(ols.conf_int().round(3))

---
## Logistic and Poisson GLMs

In [ ]:
# Logistic regression: binary outcome
logit = smf.logit("restored ~ elevation + nitrate + C(treatment)",
                   data=df).fit()
print(logit.summary().tables[1])
# Odds ratios
OR = np.exp(logit.params)
OR_ci = np.exp(logit.conf_int())
print("\nOdds Ratios:")
print(pd.concat([OR.rename('OR'), OR_ci], axis=1).round(3))

# Poisson GLM for counts
pois = smf.glm("invertcount ~ elevation + nitrate + C(treatment)",
               data=df,
               family=sm.families.Poisson()).fit()
print("\nPoisson deviance:", round(pois.deviance, 2))
print("Pearson chi2:    ", round(pois.pearson_chi2, 2))
print("Dispersion ratio:", round(pois.pearson_chi2 / pois.df_resid, 3),
      " (>1.5 suggests overdispersion -> use NegBin)")

---
## Negative Binomial for Overdispersed Counts

In [ ]:
# Negative binomial handles overdispersion
nb = smf.glm("invertcount ~ elevation + nitrate + C(treatment)",
             data=df,
             family=sm.families.NegativeBinomial()).fit()
print("Negative Binomial summary:")
print(nb.summary().tables[1])

# Compare AIC
print(f"\nAIC comparison:")
print(f"  Poisson:           {pois.aic:.2f}")
print(f"  Negative Binomial: {nb.aic:.2f}")
print("Lower AIC = better fit; NegBin should win with overdispersed counts")

# Incidence rate ratios (exponentiated NegBin coefficients)
IRR = np.exp(nb.params)
IRR_ci = np.exp(nb.conf_int())
print("\nIncidence Rate Ratios:")
print(pd.concat([IRR.rename('IRR'), IRR_ci], axis=1).round(3))

In [ ]:
# Gamma GLM: continuous positive response (e.g. nitrate concentration)
# Simulate a right-skewed positive response
biomass = rng.gamma(shape=2 + 0.005*elevation, scale=1.5, size=n)
df['biomass'] = biomass
gamma_mod = smf.glm("biomass ~ elevation + C(treatment)",
                    data=df,
                    family=sm.families.Gamma(link=sm.families.links.Log())).fit()
print("Gamma GLM (log link):")
print(gamma_mod.summary().tables[1])
print(f"\nDispersion parameter: {gamma_mod.scale:.4f}")
print("Dispersion > 1 with Gamma is expected and estimated, not a problem")

---

## Common Pitfalls

**1. Using OLS for binary or count outcomes**  
OLS applied to a 0/1 outcome can produce predicted probabilities outside [0,1] and violates homoscedasticity. OLS on count data underestimates variance at high counts. Always match the GLM family to the distributional properties of the response.

**2. Ignoring overdispersion in Poisson models**  
Poisson assumes mean = variance. When the variance is substantially larger (dispersion ratio > 1.5), Poisson standard errors are too small and p-values too optimistic. Always check the Pearson chi-squared / residual df ratio; use Negative Binomial or quasi-Poisson when overdispersed.

**3. Interpreting GLM coefficients on the link scale as effect sizes**  
Logistic regression coefficients are log-odds; Poisson/NegBin coefficients are log rate ratios. Always exponentiate to get odds ratios or incidence rate ratios before interpreting effect sizes, and report both the coefficient and the exponentiated form.

**4. Not checking residual deviance against degrees of freedom**  
For a well-fitting GLM, residual deviance / df should be approximately 1. A ratio >> 1 indicates underfitting or overdispersion; a ratio << 1 (rare) indicates overfitting. Always inspect this ratio in the summary before trusting inference.

**5. Using `C()` in formulas without verifying the reference level**  
`C(treatment)` uses the first alphanumeric level as the reference. If "control" sorts after "restored", the reference level is "restored" — the opposite of the intended contrast. Always check `model.model.data.param_names` or set the reference explicitly with `C(treatment, Treatment("control"))`.
---
*python_methods_library - Samantha McGarrigle*